# 🏫 School Bullying Analysis using the CRDC 2021–22 Dataset  
## Notebook 02 — ETL Pipeline v2.0

**Author:** Yoldas Erdem  
**Project:** Data Analytics & AI Bootcamp Capstone  
**Dataset:** Civil Rights Data Collection (CRDC) 2021–22 — Harassment & Bullying  

---

## Purpose

This notebook prepares the CRDC 2021–22 Harassment & Bullying dataset for analysis in PostgreSQL, SQL, and Tableau.

The goal of this ETL pipeline is to transform the original raw CRDC file into a clean, documented, analysis-ready dataset.

---

## Pipeline Overview

```text
Raw CRDC ZIP
      ↓
Load Harassment & Bullying CSV
      ↓
Select project variables
      ↓
Rename CRDC technical columns
      ↓
Replace reserve codes with NULL
      ↓
Validate cleaned dataset
      ↓
Export clean CSV
      ↓
Generate Data Dictionary
```

## 1. Project Configuration

This section defines reusable paths for the project.

The notebook assumes the following project structure:

```text
School_Bullying_Capstone/
│
├── data/
│   ├── raw/
│   └── processed/
│
├── documentation/
│   └── appendix/
│
└── notebooks/
    └── 02_ETL_Pipeline.ipynb
```

Using `pathlib` keeps the notebook portable and avoids hard-coded file locations.

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd

# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------
# This notebook is stored inside the notebooks/ folder.
# Therefore, Path.cwd().parent points to the project root.

project_root = Path.cwd().parent

raw_data = project_root / "data" / "raw" / "2021-22-crdc-data.zip"
clean_data = project_root / "data" / "processed" / "crdc_bullying_clean.csv"
dictionary_path = project_root / "documentation" / "appendix" / "Data_Dictionary.xlsx"

print("Project root:", project_root)
print("Raw data exists:", raw_data.exists())
print("Clean data output:", clean_data)
print("Data dictionary output:", dictionary_path)

## 2. Load Raw Dataset

The official CRDC download is provided as a ZIP archive containing multiple topical datasets.

This project only uses the school-level **Harassment and Bullying** dataset:

```text
SCH/Harassment and Bullying.csv
```

Two identifier fields are explicitly loaded as text:

- `LEAID`
- `COMBOKEY`

This prevents leading zeros from being removed.

In [ ]:
# ---------------------------------------------------------
# Load Harassment & Bullying dataset from the CRDC ZIP file
# ---------------------------------------------------------

with zipfile.ZipFile(raw_data, "r") as z:
    with z.open("SCH/Harassment and Bullying.csv") as f:
        df = pd.read_csv(
            f,
            dtype={
                "LEAID": str,
                "COMBOKEY": str
            },
            low_memory=False
        )

df.head()

## 3. Initial Data Inspection

Before transforming the dataset, the raw file is inspected to confirm:

- number of rows
- number of columns
- data types
- successful file loading

This step ensures that the expected CRDC file was loaded correctly.

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

df.info()

## 4. Select Project Variables

The Harassment & Bullying dataset contains four relevant variable groups:

| Group | Purpose |
|---|---|
| Metadata | State, district, school identifiers |
| Allegations | Reported harassment or bullying allegations |
| Reported students | Students reported as affected |
| Disciplined students | Students disciplined in relation to reported incidents |

The full Harassment & Bullying dataset is kept because all 159 variables belong to these analytical groups.

In [ ]:
# ---------------------------------------------------------
# Define metadata columns
# ---------------------------------------------------------
# These fields identify the state, district, school, and school type.

metadata_cols = [
    "LEA_STATE",
    "LEA_STATE_NAME",
    "LEAID",
    "LEA_NAME",
    "SCHID",
    "SCH_NAME",
    "COMBOKEY",
    "JJ"
]

# ---------------------------------------------------------
# Define analytical variable groups
# ---------------------------------------------------------
# These lists are created programmatically to avoid manually typing dozens of columns.

allegation_cols = [
    col for col in df.columns
    if col.startswith("SCH_HBALLEGATIONS_")
]

reported_cols = [
    col for col in df.columns
    if col.startswith("SCH_HBREPORTED_") or col.startswith("TOT_HBREPORTED_")
]

disciplined_cols = [
    col for col in df.columns
    if col.startswith("SCH_HBDISCIPLINED_") or col.startswith("TOT_HBDISCIPLINED_")
]

print(f"Metadata columns:     {len(metadata_cols)}")
print(f"Allegation columns:   {len(allegation_cols)}")
print(f"Reported columns:     {len(reported_cols)}")
print(f"Disciplined columns:  {len(disciplined_cols)}")

In [ ]:
# ---------------------------------------------------------
# Build the project dataset
# ---------------------------------------------------------
# The selected columns form the base dataset for cleaning and later analysis.

selected_columns = (
    metadata_cols
    + allegation_cols
    + reported_cols
    + disciplined_cols
)

project_df = df[selected_columns].copy()

print(f"Project dataset shape: {project_df.shape}")
project_df.head()

## 5. Rename Variables

The original CRDC column names are technically accurate but difficult to use in SQL and Tableau.

Examples:

```text
SCH_HBALLEGATIONS_SEX
TOT_HBREPORTED_SEX_M
SCH_HBDISCIPLINED_DIS_504_F
```

This step converts them into readable `snake_case` names, while preserving their meaning.

Examples:

```text
allegation_sex
total_reported_sex_male
disciplined_disability_section_504_female
```

In [ ]:
# ---------------------------------------------------------
# Create working copy for cleaned dataset
# ---------------------------------------------------------

clean_df = project_df.copy()

# ---------------------------------------------------------
# Manual renaming for metadata columns
# ---------------------------------------------------------

metadata_rename = {
    "LEA_STATE": "state_code",
    "LEA_STATE_NAME": "state",
    "LEAID": "district_id",
    "LEA_NAME": "district",
    "SCHID": "school_id",
    "SCH_NAME": "school",
    "COMBOKEY": "school_key",
    "JJ": "juvenile_justice_school"
}

# ---------------------------------------------------------
# CRDC abbreviation mappings
# ---------------------------------------------------------
# These dictionaries translate repeated CRDC codes into readable labels.

category_map = {
    "SEX": "sex",
    "ORI": "orientation",
    "RAC": "race",
    "DIS": "disability",
    "REL": "religion",
    "GI": "gender_identity"
}

student_group_map = {
    "HI": "hispanic",
    "AM": "american_indian",
    "AS": "asian",
    "HP": "pacific_islander",
    "BL": "black",
    "WH": "white",
    "TR": "multi_race",
    "EL": "english_learner",
    "IDEA": "idea",
    "504": "section_504"
}

sex_map = {
    "M": "male",
    "F": "female"
}

In [ ]:
def rename_crdc_column(col):
    """
    Convert original CRDC variable names into readable snake_case names.

    Examples
    --------
    SCH_HBALLEGATIONS_SEX
        -> allegation_sex

    TOT_HBREPORTED_SEX_M
        -> total_reported_sex_male

    SCH_HBDISCIPLINED_DIS_504_F
        -> disciplined_disability_section_504_female
    """

    # Metadata columns are handled manually.
    if col in metadata_rename:
        return metadata_rename[col]

    parts = col.split("_")

    # Allegation columns.
    if col.startswith("SCH_HBALLEGATIONS_"):
        category_code = parts[-1]
        category = category_map.get(category_code, category_code.lower())
        return f"allegation_{category}"

    # Reported student columns.
    if "HBREPORTED" in col:
        prefix = "total_reported" if col.startswith("TOT_") else "reported"
        remaining_parts = parts[2:]

        readable_parts = []
        for part in remaining_parts:
            if part in category_map:
                readable_parts.append(category_map[part])
            elif part in student_group_map:
                readable_parts.append(student_group_map[part])
            elif part in sex_map:
                readable_parts.append(sex_map[part])
            else:
                readable_parts.append(part.lower())

        return prefix + "_" + "_".join(readable_parts)

    # Disciplined student columns.
    if "HBDISCIPLINED" in col:
        prefix = "total_disciplined" if col.startswith("TOT_") else "disciplined"
        remaining_parts = parts[2:]

        readable_parts = []
        for part in remaining_parts:
            if part in category_map:
                readable_parts.append(category_map[part])
            elif part in student_group_map:
                readable_parts.append(student_group_map[part])
            elif part in sex_map:
                readable_parts.append(sex_map[part])
            else:
                readable_parts.append(part.lower())

        return prefix + "_" + "_".join(readable_parts)

    # Fallback for unexpected columns.
    return col.lower()

In [ ]:
# ---------------------------------------------------------
# Apply column renaming
# ---------------------------------------------------------

clean_df = clean_df.rename(
    columns={col: rename_crdc_column(col) for col in clean_df.columns}
)

print(f"Cleaned dataset shape after renaming: {clean_df.shape}")
clean_df.head()

## 6. Quality Checks Before Cleaning

Before replacing reserve codes, this step checks:

- dataset shape
- missing values
- duplicate rows
- first cleaned column names

At this stage, missing values are expected to be zero because CRDC reserve codes are still stored as numeric values.

In [ ]:
print(f"Rows: {clean_df.shape[0]:,}")
print(f"Columns: {clean_df.shape[1]}")

print("\nMissing values before reserve-code cleaning:")
print(clean_df.isna().sum().sum())

print("\nDuplicate rows:")
print(clean_df.duplicated().sum())

print("\nFirst 30 cleaned columns:")
print(clean_df.columns.tolist()[:30])

## 7. Replace CRDC Reserve Codes with NULL

The CRDC uses negative reserve codes for administrative cases such as missing, skipped, suppressed, or not applicable values.

These values are **not real counts** and should not be included in numerical analysis.

The following reserve codes are converted to `NULL`:

| Reserve Code | Meaning |
|---|---|
| -3 | Skip logic or processing failure |
| -4 | Missing optional data |
| -5 | Action plan / quick plan |
| -6 | Force certified |
| -9 | Not applicable / skipped |
| -12 | Suppressed for privacy protection |
| -13 | Missing DIND skip logic |

Only measurement columns are cleaned. Metadata fields remain unchanged.

In [ ]:
# ---------------------------------------------------------
# Replace CRDC reserve codes with NULL
# ---------------------------------------------------------

reserve_codes = [-3, -4, -5, -6, -9, -12, -13]

metadata_columns = [
    "state_code",
    "state",
    "district_id",
    "district",
    "school_id",
    "school",
    "school_key",
    "juvenile_justice_school"
]

measure_columns = [
    col for col in clean_df.columns
    if col not in metadata_columns
]

clean_df[measure_columns] = clean_df[measure_columns].replace(reserve_codes, pd.NA)

print("Reserve codes converted to NULL.")
print("Total NULL values after cleaning:", clean_df[measure_columns].isna().sum().sum())

## 8. Export Clean Dataset

The cleaned dataset is exported as a CSV file.

This file will later be imported into PostgreSQL through DBeaver and connected to Tableau for dashboard development.

In [ ]:
# ---------------------------------------------------------
# Export cleaned dataset
# ---------------------------------------------------------

clean_df.to_csv(clean_data, index=False)

print("Clean dataset exported successfully.")
print(clean_data)

## 9. Validate Exported Dataset

A professional ETL process should not assume that an export worked correctly.

This step reloads the exported CSV and validates:

- row and column count
- absence of reserve codes
- presence of NULL values
- cleaned column names
- preservation of identifier fields as text

In [ ]:
# ---------------------------------------------------------
# Reload exported CSV for validation
# ---------------------------------------------------------

validation_df = pd.read_csv(
    clean_data,
    dtype={
        "district_id": str,
        "school_key": str
    },
    low_memory=False
)

print(f"Validation shape: {validation_df.shape}")

print("\nReserve code check:")
for code in reserve_codes:
    print(code, (validation_df == code).sum().sum())

print("\nTotal NULL values:")
print(validation_df.isna().sum().sum())

validation_df.head()

## 10. Generate Data Dictionary

The data dictionary documents the cleaned analytical dataset.

It includes:

- variable category
- original CRDC variable name
- cleaned variable name
- plain-English description
- interpretation notes

This file supports SQL analysis, Tableau dashboard development, and final project documentation.

In [ ]:
# ---------------------------------------------------------
# Create base data dictionary
# ---------------------------------------------------------

data_dictionary = pd.DataFrame({
    "Original Variable": project_df.columns,
    "Clean Variable": clean_df.columns
})


def get_category(original_col):
    """Assign each variable to a broad analytical category."""

    if original_col in metadata_cols:
        return "Metadata"
    if "ALLEGATION" in original_col.upper():
        return "Allegation"
    if "REPORTED" in original_col.upper():
        return "Reported"
    if "DISCIPLINED" in original_col.upper():
        return "Disciplined"
    return "Other"


data_dictionary.insert(
    0,
    "Category",
    data_dictionary["Original Variable"].apply(get_category)
)

data_dictionary.head()

In [ ]:
# ---------------------------------------------------------
# Human-readable labels for descriptions
# ---------------------------------------------------------

basis_map = {
    "sex": "sex-based",
    "orientation": "sexual-orientation-based",
    "race": "race- or ethnicity-based",
    "disability": "disability-based",
    "religion": "religion-based",
    "atag": "religion-based allegations targeting atheists or agnostics",
    "budd": "religion-based allegations targeting Buddhist students",
    "cath": "religion-based allegations targeting Catholic students",
    "east": "religion-based allegations targeting Eastern Orthodox Christian students",
    "hind": "religion-based allegations targeting Hindu students",
    "islm": "religion-based allegations targeting Muslim students",
    "jwit": "religion-based allegations targeting Jehovah's Witness students",
    "jwsh": "religion-based allegations targeting Jewish students",
    "morm": "religion-based allegations targeting Mormon / Latter-day Saints students",
    "multi": "religion-based allegations targeting multiple religious groups",
    "othchrn": "religion-based allegations targeting other Christian groups",
    "othrel": "religion-based allegations targeting other religions",
    "prot": "religion-based allegations targeting Protestant students",
    "sikh": "religion-based allegations targeting Sikh students"
}

group_map = {
    "hispanic": "Hispanic",
    "american_indian": "American Indian or Alaska Native",
    "asian": "Asian",
    "pacific_islander": "Native Hawaiian or Pacific Islander",
    "black": "Black or African American",
    "white": "White",
    "multi_race": "two or more races",
    "english_learner": "English learner",
    "idea": "IDEA disability",
    "section_504": "Section 504 disability",
    "male": "male",
    "female": "female"
}

metadata_descriptions = {
    "state_code": "Two-letter abbreviation of the U.S. state where the school is located.",
    "state": "Full name of the U.S. state where the school is located.",
    "district_id": "Unique identifier assigned to the Local Education Agency / school district.",
    "district": "Official name of the Local Education Agency / school district.",
    "school_id": "Unique identifier assigned to the school within the district.",
    "school": "Official name of the school.",
    "school_key": "Unique identifier created by combining district ID and school ID.",
    "juvenile_justice_school": "Indicates whether the school is a juvenile justice facility."
}

metadata_notes = {
    "state_code": "Geographic identifier.",
    "state": "Geographic identifier.",
    "district_id": "Store as text to preserve leading zeros.",
    "district": "School district / LEA name.",
    "school_id": "School identifier.",
    "school": "School name.",
    "school_key": "Primary school-level key; store as text to preserve leading zeros.",
    "juvenile_justice_school": "Yes/No field."
}

In [ ]:
def pretty_basis(value):
    """Return a readable description of the harassment or bullying basis."""
    return basis_map.get(value, value.replace("_", " ") + "-based")


def pretty_group(parts):
    """Convert demographic code parts into a readable student group label."""
    labels = [group_map.get(part, part.replace("_", " ")) for part in parts]
    return " ".join(labels)


def create_description(clean_col):
    """Generate a plain-English description for a cleaned variable name."""

    if clean_col in metadata_descriptions:
        return metadata_descriptions[clean_col]

    if clean_col.startswith("allegation_"):
        basis = clean_col.replace("allegation_", "")
        label = basis_map.get(basis, basis.replace("_", " "))

        religion_breakdowns = [
            "atag", "budd", "cath", "east", "hind", "islm", "jwit",
            "jwsh", "morm", "multi", "othchrn", "othrel", "prot", "sikh"
        ]

        if basis in religion_breakdowns:
            return f"Number of reported {label}."

        return f"Number of reported harassment or bullying allegations based on {label.replace('-based', '')}."

    if clean_col.startswith("total_reported_"):
        rest = clean_col.replace("total_reported_", "")
        parts = rest.split("_")
        basis = parts[0]
        group = pretty_group(parts[1:])
        return f"Total number of {group} students reported as affected by {pretty_basis(basis)} harassment or bullying."

    if clean_col.startswith("reported_"):
        rest = clean_col.replace("reported_", "")
        parts = rest.split("_")
        basis = parts[0]
        group = pretty_group(parts[1:])
        return f"Number of {group} students reported as affected by {pretty_basis(basis)} harassment or bullying."

    if clean_col.startswith("total_disciplined_"):
        rest = clean_col.replace("total_disciplined_", "")
        parts = rest.split("_")
        basis = parts[0]
        group = pretty_group(parts[1:])
        return f"Total number of {group} students disciplined in relation to {pretty_basis(basis)} harassment or bullying."

    if clean_col.startswith("disciplined_"):
        rest = clean_col.replace("disciplined_", "")
        parts = rest.split("_")
        basis = parts[0]
        group = pretty_group(parts[1:])
        return f"Number of {group} students disciplined in relation to {pretty_basis(basis)} harassment or bullying."

    return "Description to be reviewed."


def create_notes(clean_col):
    """Generate interpretation notes for variables."""

    if clean_col in metadata_notes:
        return metadata_notes[clean_col]

    if clean_col.startswith("allegation_"):
        return "School-level allegation count."

    if clean_col.startswith("reported_") or clean_col.startswith("total_reported_"):
        return "Student count; not an individual incident-level record."

    if clean_col.startswith("disciplined_") or clean_col.startswith("total_disciplined_"):
        return "Student count; disciplinary counts are not linked one-to-one with individual allegations."

    return ""

In [ ]:
# ---------------------------------------------------------
# Apply descriptions and export data dictionary
# ---------------------------------------------------------

data_dictionary["Description"] = data_dictionary["Clean Variable"].apply(create_description)
data_dictionary["Notes"] = data_dictionary["Clean Variable"].apply(create_notes)

data_dictionary.to_excel(dictionary_path, index=False)

print("Data dictionary exported successfully.")
print(dictionary_path)
print(data_dictionary.shape)

data_dictionary.head(20)

## 11. Final ETL Summary

The ETL pipeline is complete.

### Outputs created

- `data/processed/crdc_bullying_clean.csv`
- `documentation/appendix/Data_Dictionary.xlsx`

The cleaned dataset is now ready for the next project phase:

```text
PostgreSQL → SQL Analysis → Tableau Dashboard
```

In [ ]:
print("ETL pipeline completed successfully.")

print("\nFinal dataset:")
print(f"Rows: {clean_df.shape[0]:,}")
print(f"Columns: {clean_df.shape[1]}")

print("\nOutput files:")
print("Clean CSV exists:", clean_data.exists())
print("Data dictionary exists:", dictionary_path.exists())